# 04 - Embeddings

### Imports

In [1]:
# imports
import pandas as pd
import re
import os
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from transformers import logging as hf_logging

import torch
from transformers import AutoTokenizer
# from adapters import AutoAdapterModel


hf_logging.set_verbosity_error()


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data

Deduplicated dgf created in 03_deduplication

In [4]:
dgf_dedup = pd.read_parquet('/content/drive/MyDrive/thesis/03_dgf_dedup.parquet')

DRP corresponding records from dgf created in 02_edg_dgf


In [5]:
drp = pd.read_parquet('/content/drive/MyDrive/thesis/02_03_drp_in_dgf_formodeling.parquet')

# Embeddings

In [6]:
def pick_device():
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'

device = pick_device()
print(device)

cuda


### all-MiniLM-L6-v2
https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [ ]:
# allminilm_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# def add_abstract_embeddings(df, name, model, text_col='text', id_col='doi', batch_size=32):
#     print(f'Extracting text embeddings for {name}...')

#     texts = df[text_col].fillna('').astype(str).tolist()

#     vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

#     vectors = np.asarray(vectors)

#     np.savez(f'embeddings/{name}_allminilm.npz', doi=df[id_col].values, embeddings=vectors)

#     print(f'Saved: embeddings/{name}_allminilm.npz')

In [ ]:
# for name, df in dfs.items():
#     add_abstract_embeddings(df, name, allminilm_model)

Extracting title embeddings for comp...


Batches:   0%|          | 0/483 [00:00<?, ?it/s]

Saved: embeddings/comp_allminilm.npz
Extracting title embeddings for df_aff...


Batches:   0%|          | 0/479 [00:00<?, ?it/s]

Saved: embeddings/df_aff_allminilm.npz


### nomic-embed-text-v1.5
https://huggingface.co/nomic-ai/nomic-embed-text-v1.5



In [7]:
nomic_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True, device='cuda').half()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [8]:
nomic_model.max_seq_length


8192

In [9]:
tok = nomic_model.tokenizer
sample = (dgf_dedup['description'].fillna('').astype(str).sample(min(5000, len(dgf_dedup)), random_state=0).tolist())

lens = np.array([len(tok.encode(s)) for s in sample])
print(f'desc tokens — median {int(np.median(lens))}  p95 {int(np.percentile(lens,95))}  '
      f'p99 {int(np.percentile(lens,99))}  max {int(lens.max())}  >2048 {(lens>2048).mean():.1%}')


desc tokens — median 174  p95 516  p99 965  max 7669  >2048 0.1%


In [10]:
nomic_model.max_seq_length = 1024

In [11]:
PREFIX = 'classification: '
def embed(col, df):
    texts = [PREFIX + t for t in df[col].fillna('').astype(str).tolist()]
    return nomic_model.encode(texts, batch_size=128, show_progress_bar=True,
                              normalize_embeddings=True, convert_to_numpy=True)

#### DGF

In [12]:
dgf_dedup.columns

Index(['title', 'description', 'publisher', 'identifier', 'slug',
       'has_spatial', 'popularity', 'last_harvested_date', 'keyword', 'theme',
       'org_name', 'org_type', 'org_id', 'org_slug', 'org_description',
       'dcat_issued', 'dcat_modified', 'dcat_access_level',
       'dcat_access_comment', 'dcat_landing_page', 'dcat_language',
       'dcat_temporal', 'dcat_spatial', 'dcat_bureau_code',
       'dcat_program_code', 'dcat_periodicity', 'dcat_license', 'dcat_rights',
       'title_norm', 'desc_len', 'desc_norm', 'title_topic'],
      dtype='object')

In [14]:
desc_vecs  = embed('description', dgf_dedup )
title_vecs = embed('title', dgf_dedup)

dgf_dedup['desc_embeddings_nomic'] = [v.tolist() for v in desc_vecs]
dgf_dedup['title_embeddings_nomic'] = [v.tolist() for v in title_vecs]
dgf_dedup.to_parquet('/content/drive/MyDrive/thesis/03_dgf_dedup_nomic.parquet', index=False)

Batches:   0%|          | 0/2103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2103 [00:00<?, ?it/s]

#### DRP

In [15]:
drp.columns

Index(['title', 'description', 'publisher', 'identifier', 'slug',
       'has_spatial', 'popularity', 'last_harvested_date', 'keyword', 'theme',
       'org_name', 'org_type', 'org_id', 'org_slug', 'org_description',
       'dcat_issued', 'dcat_modified', 'dcat_access_level',
       'dcat_access_comment', 'dcat_landing_page', 'dcat_language',
       'dcat_temporal', 'dcat_spatial', 'dcat_bureau_code',
       'dcat_program_code', 'dcat_periodicity', 'dcat_license', 'dcat_rights',
       'title_norm'],
      dtype='object')

In [17]:
drp_desc_vecs  = embed('description', drp)
drp_title_vecs = embed('title', drp)

drp['desc_embeddings_nomic']  = [v.tolist() for v in drp_desc_vecs]
drp['title_embeddings_nomic'] = [v.tolist() for v in drp_title_vecs]
drp.to_parquet('/content/drive/MyDrive/thesis/03_drp_in_dgf_nomic.parquet', index=False)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]